[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/byu-matrix-lab/torchlingo/blob/main/docs/docs/course/lecture-19-post-editing.ipynb)

# Lecture 19: how much did the post-editor change?

Much professional translation is now **post-editing**: a translator corrects a machine's output
rather than translating from scratch, and is often paid by how much there was to correct. The
usual measure of that is an edit distance: the fewest word changes that turn the MT output into
the post-edit, divided by the post-edit's length. Measured against a human post-edit, it is
called **HTER**, human-targeted translation edit rate. In this notebook you post-edit five MT
outputs yourself, measure your edits two ways, set them beside a professional translator's
post-edits of the same outputs, and then look for what the number cannot see.

By the end you will be able to:

1. compute the word-level edit distance between an MT output and its post-edit, and turn it into
   an edit rate;
2. explain why TER counts a moved phrase as one edit, and when that changes the number;
3. explain why two correct post-edits of the same output can get very different numbers;
4. say what an edit rate cannot measure: how much an edit changes the meaning, and how long it
   took to find.

These serve Lecture 19's outcome *explain how edit distance measures post-editing effort, and
what it misses*.

Work in pairs. Two cells stop the notebook until you write in them: your post-edits, and a
prediction. Everything runs on a CPU in seconds, so no GPU is needed. About 15 minutes.

**Running in Google Colab?** Run the cells in order. There is nothing to uncomment.

<!-- GENERATED:purpose - change the torchlingo metadata, not this cell -->

**CS 479 course notebook** — the in-class activity for Lecture 19 (*MT in Use: Applications, Jobs and Post-Editing*).

In [ ]:
# Install TorchLingo in Colab. Nothing to uncomment. Pip's "you may need to restart the
# kernel" does not apply to a first install; if the install fails, the import below stops here.
import sys
if "google.colab" in sys.modules:
    %pip install --quiet "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
    # pip keeps a TorchLingo already in this session when its version number matches, even if
    # it is older code; this swaps in the course branch's, leaving the dependencies alone.
    %pip install --quiet --force-reinstall --no-deps "torchlingo @ git+https://github.com/byu-matrix-lab/torchlingo@course"
import torchlingo

In [ ]:
from torchlingo.colab import setup

env = setup()

import re

from sacrebleu.metrics import TER

## Step 1: five machine translations, and what they should say

The sentences come from **MLQE-PE** (Fomicheva and colleagues, 2020), a public dataset built
to study post-editing. Estonian Wikipedia sentences were translated into English by a neural MT
system, and professional translators post-edited every output. You will see their post-edits in
Step 5, not before.

You do not need any Estonian. Under each MT output is **what the sentence means**: a translation
a different translator made from the Estonian, without seeing the MT.

In [ ]:
# Five sentences from MLQE-PE's Estonian-English test set (CC0). For each: the MT output; a
# translation made independently from the Estonian, for the meaning; a professional translator's
# post-edit of the MT output; and that post-edit's HTER, as the dataset publishes it.
SENTENCES = [
    {
        "mt": "Johannes tells him that Jaani has a new carrot.",
        "meaning": "Johannes tells her that Jaanus has a new girlfriend.",
        "professional": "Johannes tells him that Jaanus has a new girlfriend.",
        "published_hter": 0.20,
    },
    {
        "mt": "The interviews looked at around 90 million American TV viewers.",
        "meaning": "About 90 million American TV viewers watched the interview.",
        "professional": "The interview was watched by around 90 million American TV viewers.",
        "published_hter": 0.33,
    },
    {
        "mt": "The idea changed owners and sold to 1783 Friedrich Wambmann.",
        "meaning": "The manor changed owners and was sold to Friedrich Wartmann in 1783.",
        "professional": "The manor changed owners and was sold in 1783 to Friedrich Wartmann.",
        "published_hter": 0.38,
    },
    {
        "mt": "It called for peace in Mali and Nigeria on 25 December 2012.",
        "meaning": "He called for peace in Mali and Nigeria on December 25, 2012.",
        "professional": "On 25 December 2012, he called for peace in Mali and Nigeria.",
        "published_hter": 0.21,
    },
    {
        "mt": "It is not easy to attack this strongly entrenched position.",
        "meaning": "This highly secured position is not easy to attack.",
        "professional": "This heavily fortified position is not easy to attack.",
        "published_hter": 0.60,
    },
]

for number, sentence in enumerate(SENTENCES, 1):
    print(f"{number}. MT:      {sentence['mt']}")
    print(f"   Meaning: {sentence['meaning']}")
    print()

## Step 2: post-edit them yourselves

`POST_EDITS` below starts as a copy of the five MT outputs. Edit each string in place, the way a
post-editor works, so that it says what the meaning line says, in correct English.

- **Change as little as you can.** The job is to fix the MT output, not to rewrite it into the
  meaning line's wording. Professional post-editors are given the same instruction.
- **If an output is already correct, leave it as it is.** Correct means right in meaning and in
  English, not worded as you would have worded it.
- **Agree with your partner** on each edit before you make it.

The cell stops until you have changed something.

In [ ]:
POST_EDITS = [
    "Johannes tells him that Jaani has a new carrot.",
    "The interviews looked at around 90 million American TV viewers.",
    "The idea changed owners and sold to 1783 Friedrich Wambmann.",
    "It called for peace in Mali and Nigeria on 25 December 2012.",
    "It is not easy to attack this strongly entrenched position.",
]

assert len(POST_EDITS) == len(SENTENCES), "Keep one post-edit for each of the five outputs."
assert POST_EDITS != [s["mt"] for s in SENTENCES], (
    "Edit the MT outputs in POST_EDITS, then run this cell again."
)

## Step 3: predict, then count

An **edit rate** is the number of word edits divided by the number of words in the post-edit, so
0.25 means one edit for every four words. **Before you count**, write in `PREDICTION`: which of
the five will have your highest edit rate, and roughly what it will be? The notebook stops at
this cell until you write it.

In [ ]:
PREDICTION = ""   # <- which sentence, and roughly what rate
assert PREDICTION.strip(), "Write your prediction in PREDICTION, then run this cell again."

The edits are counted in words, so first each sentence becomes a list of words, lowercased, with
each punctuation mark a word of its own: `"On 25 December 2012, he"` is `on`, `25`, `december`,
`2012`, `,` and `he`. That is how the standard HTER tools count, too.

Then the **edit distance**: the fewest insertions, deletions and substitutions of words that turn
the MT output into the post-edit. It is the dynamic program you have met as Levenshtein distance,
over words instead of letters. `d[i][j]` is the fewest edits that turn the MT's first `i` words
into the post-edit's first `j`, and each cell takes the cheapest of three ways to get there:
delete the MT's word `i`, insert the post-edit's word `j`, or line the two words up, which costs
nothing when they match and one substitution when they do not.

In [ ]:
def words(text):
    """Lowercased words and punctuation marks: the units edits are counted in."""
    return re.findall(r"\w+(?:['’-]\w+)*|[^\w\s]", text.lower())


def edit_distance(mt, post_edit):
    """The fewest word insertions, deletions and substitutions that turn mt into post_edit."""
    a, b = words(mt), words(post_edit)
    # d[i][j]: the fewest edits that turn a's first i words into b's first j.
    d = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(len(a) + 1):
        d[i][0] = i   # delete all i words
    for j in range(len(b) + 1):
        d[0][j] = j   # insert all j words
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            d[i][j] = min(
                d[i - 1][j] + 1,                            # delete a's word i
                d[i][j - 1] + 1,                            # insert b's word j
                d[i - 1][j - 1] + (a[i - 1] != b[j - 1]),   # keep a match, or substitute
            )
    return d[len(a)][len(b)]


def edit_rate(mt, post_edit):
    """Edits per word of the post-edit."""
    return edit_distance(mt, post_edit) / len(words(post_edit))


# A case to check in your head: one substitution (cat to dog) and one insertion (down).
print("the cat sat -> the dog sat down:", edit_distance("the cat sat", "the dog sat down"), "edits")
print()
for number, (sentence, mine) in enumerate(zip(SENTENCES, POST_EDITS), 1):
    edits, length = edit_distance(sentence["mt"], mine), len(words(mine))
    print(f"{number}. {edits:2d} edits / {length:2d} words = {edit_rate(sentence['mt'], mine):.2f}")
print()
print("You predicted:", PREDICTION.strip())

## Step 4: TER, where a moved phrase is one edit

Suppose a post-editor moves a four-word phrase from the end of a sentence to the front. That is
one decision, one drag of the mouse. The edit distance above has no move, so it charges four
deletions and four insertions, or substitutions in their place.

**TER**, translation edit rate (Snover and colleagues, 2006), adds a fourth edit to the three: a
**shift**, which moves a run of adjacent words anywhere in the sentence for the price of one.
HTER is TER measured against a human post-edit; it is what MLQE-PE publishes. TER is in
`sacrebleu`, the library that scores BLEU and chrF, and the cell below gives it the same words
the edit distance used.

In [ ]:
ter = TER()   # insertions, deletions, substitutions, and shifts


def ter_rate(mt, post_edit):
    """TER of mt against post_edit, over the same words as edit_rate."""
    hypothesis, reference = " ".join(words(mt)), " ".join(words(post_edit))
    return ter.sentence_score(hypothesis, [reference]).score / 100


print("       edit    TER")
for number, (sentence, mine) in enumerate(zip(SENTENCES, POST_EDITS), 1):
    print(f"{number}.  {edit_rate(sentence['mt'], mine):6.2f} {ter_rate(sentence['mt'], mine):6.2f}")

Where the two columns agree, TER found no move worth making in your edits. Where TER is lower,
it found one: words you deleted in one place and typed in another.

## Step 5: a professional's post-edits of the same outputs

Here are the post-edits of the professional translator, who read the Estonian, and both
measures for each, beside yours. The last column is the HTER that MLQE-PE publishes for the professional's
post-edit.

In [ ]:
for number, (sentence, mine) in enumerate(zip(SENTENCES, POST_EDITS), 1):
    print(f"{number}. MT:           {sentence['mt']}")
    print(f"   yours:        {mine}")
    print(f"   professional: {sentence['professional']}")
    print()

print("         yours          professional    published")
print("       edit    TER      edit    TER        HTER")
for number, (sentence, mine) in enumerate(zip(SENTENCES, POST_EDITS), 1):
    mt, theirs = sentence["mt"], sentence["professional"]
    print(
        f"{number}.  {edit_rate(mt, mine):6.2f} {ter_rate(mt, mine):6.2f}  "
        f"  {edit_rate(mt, theirs):6.2f} {ter_rate(mt, theirs):6.2f}"
        f"      {sentence['published_hter']:6.2f}"
    )

**Sentence 4: a move.** The professional moved *on 25 December 2012* to the front. The edit
distance charges that as ten edits in fourteen words, 0.71; TER charges one shift, plus *it* to
*he* and a comma, 0.21, which is the published HTER. If you only changed *It* to *He*, your
post-edit is as correct as theirs, with one edit.

**Sentence 5: an output that needed nothing.** The MT output says what the sentence means, in
good English. The professional reordered it anyway, to follow the Estonian, which puts *this
strongly fortified position* first, and scored 0.60. Two correct post-edits of the same output,
and very different numbers: HTER measures the post-editor as well as the MT. That is why
post-editors are told to change as little as they can, and why an HTER from one post-editor is
not comparable with an HTER from another.

**Sentence 1: a choice the English forces.** Estonian *talle* means *to him* or *to her*; one
word does for both. The meaning line chose *her* and the professional kept the MT's *him*. Each
is a correct translation, and whichever you chose cost you an edit or did not.

Compare the professional's numbers with yours, sentence by sentence. Where are they furthest
apart, and which of you changed more?

## Step 6: what the number misses

Two more outputs, each with a professional's post-edit from MLQE-PE: one from Estonian, and one
from Nepali. Each post-editor changed exactly one word.

In [ ]:
ONE_WORD = [
    (   # Estonian-English, published HTER 0.07
        "The British Prime Minister, Tony Blair, also took part in the final event.",
        "British Prime Minister, Tony Blair, also took part in the final event.",
    ),
    (   # Nepali-English, published HTER 0.07
        "It is obvious that they are affecting the growth of the disease.",
        "It is not obvious that they are affecting the growth of the disease.",
    ),
]

for mt, post_edit in ONE_WORD:
    print(f"MT:        {mt}")
    print(f"post-edit: {post_edit}")
    print(f"edit {edit_rate(mt, post_edit):.2f}   TER {ter_rate(mt, post_edit):.2f}")
    print()

The two get the same number. The first edit drops an article, and the MT output meant the same
without it. The second reverses the sentence: the Nepali says that it is *unclear* whether they
affect the disease, and the MT output says it is obvious. A reader who trusted that output would
believe the opposite of the source, and an edit rate of 0.07 says the output was nearly right.

**An edit rate counts edits; it does not weigh them.** Nor does it see the work behind an edit.
In sentence 1, *carrot* to *girlfriend* is one substitution, as cheap as dropping *The*. Seeing
that *carrot* is wrong is easy; knowing that the word should be *girlfriend* took the Estonian,
*pruut*. And a post-editor who reads an output twice, checks a name, and leaves it unchanged has
an edit rate of 0, for work that took real time.

Find the same thing in your own post-edits: the edit that mattered most to the meaning, and the
one that mattered least. Did they cost the same?

In [ ]:
MOST_AND_LEAST = ""   # <- your edit that mattered most, the one that mattered least, and what each cost

print("You wrote:", MOST_AND_LEAST.strip() or "(nothing yet)")

## Report back

Discuss with your partner, then with the class: **when is edit distance a fair measure of
post-editing effort, and when is it not?** Between you, cover:

- what a shift changes, and a kind of edit that TER still charges more than one decision for;
- why two post-editors' HTER for the same MT output can differ, and what that means for
  comparing two MT systems by HTER;
- an edit that costs one word and changes everything, and work that costs no edits at all.

A Final Project that has someone post-edit its system's output can report HTER: `ter_rate` above,
with the post-edit as the reference, is it.

*Data:* Marina Fomicheva, Shuo Sun, Erick Fonseca, Frédéric Blain, Vishrav Chaudhary, Francisco
Guzmán, Nina Lopatina, Lucia Specia, and André F. T. Martins, "MLQE-PE: A Multilingual Quality
Estimation and Post-Editing Dataset", arXiv:2010.04480, 2020, released under CC0
([github.com/sheffieldnlp/mlqe-pe](https://github.com/sheffieldnlp/mlqe-pe)).